# 05 — Attention causale en PyTorch

On passe de l’explication numérique du notebook 04 à un module PyTorch réutilisable. Le code progresse par petites étapes : tenseurs, projections Q/K/V, masque causal, puis classe d’attention.

Les valeurs d’entrée sont synthétiques. Les projections sont initialisées aléatoirement : aucun texte AtlasIA n’est chargé et aucun modèle n’est entraîné dans ce notebook.

## Objectif et formes des tenseurs

Le module reçoit une entrée de forme **(B, T, d_in)** : B séquences, T positions et d_in caractéristiques par position. Il renvoie un vecteur de contexte à chaque position.

Cette étape implémente uniquement l’attention causale. Le multi-têtes, les connexions résiduelles, la normalisation et le MLP viendront ensuite.

In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(123)

# Entrée numérique synthétique : B=2 séquences, T=4 positions, d_in=3.
x = torch.tensor([
    [[0.43, 0.15, 0.89], [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64], [0.22, 0.58, 0.33]],
    [[0.77, 0.25, 0.10], [0.05, 0.80, 0.55],
     [0.31, 0.46, 0.72], [0.62, 0.12, 0.40]],
])

print("x :", tuple(x.shape))  # (B, T, d_in)

## 1. Projeter les entrées en Q, K et V

Trois couches linéaires apprenables produisent les requêtes **Q**, les clés **K** et les valeurs **V**. Les mêmes couches sont appliquées à toutes les positions du batch.

In [ ]:
d_in = x.shape[-1]
d_out = 4

W_query = nn.Linear(d_in, d_out, bias=False)
W_key = nn.Linear(d_in, d_out, bias=False)
W_value = nn.Linear(d_in, d_out, bias=False)

queries = W_query(x)
keys = W_key(x)
values = W_value(x)

print("Q, K, V :", tuple(queries.shape), tuple(keys.shape), tuple(values.shape))

## 2. Scores et poids d’attention

Le produit QKᵀ compare chaque position aux autres. On divise les scores par √dₖ, puis le softmax transforme chaque ligne en poids qui somment à 1.

In [ ]:
scores = queries @ keys.transpose(-2, -1)
scores = scores / math.sqrt(d_out)
weights = torch.softmax(scores, dim=-1)

print("Scores :", tuple(scores.shape))  # (B, T, T)
print("Poids pour la séquence 0 :\n", weights[0].round(decimals=3))
assert torch.allclose(weights.sum(dim=-1), torch.ones(x.shape[0], x.shape[1]))

## 3. Masque causal

Pour prédire le prochain token, une position ne doit pas voir les positions futures. On remplace leurs scores par −∞ **avant** le softmax ; elles reçoivent donc un poids nul.

In [ ]:
batch_size, sequence_length, _ = x.shape
causal_mask = torch.tril(torch.ones(sequence_length, sequence_length, dtype=torch.bool))

masked_scores = scores.masked_fill(~causal_mask, float("-inf"))
causal_weights = torch.softmax(masked_scores, dim=-1)
context = causal_weights @ values

print("Masque :\n", causal_mask.int())
print("Poids masqués :\n", causal_weights[0].round(decimals=3))
print("Contexte :", tuple(context.shape))  # (B, T, d_out)
assert torch.count_nonzero(causal_weights.masked_select(~causal_mask[None, :, :])) == 0
assert torch.allclose(causal_weights.sum(dim=-1), torch.ones(batch_size, sequence_length))

## 4. Regrouper les opérations dans une classe

Le masque est un buffer : il suit le module lorsqu’on le déplace sur CPU ou GPU, mais n’est pas appris. Le dropout sur les poids est actif en mode entraînement et désactivé par `eval()`.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout=0.0):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(context_length, context_length, dtype=torch.bool))
        self.register_buffer("causal_mask", mask)

    def forward(self, x):
        # x : (B, T, d_in)
        _, sequence_length, _ = x.shape
        if sequence_length > self.causal_mask.size(0):
            raise ValueError("La séquence dépasse context_length.")

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        scores = queries @ keys.transpose(-2, -1)
        scores = scores / math.sqrt(self.d_out)
        allowed = self.causal_mask[:sequence_length, :sequence_length]
        scores = scores.masked_fill(~allowed, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        context = weights @ values
        return context, weights

In [ ]:
attention = CausalSelfAttention(d_in=3, d_out=8, context_length=8, dropout=0.1)
attention.eval()
output, weights = attention(x)

print("Sortie :", tuple(output.shape))
print("Poids :", tuple(weights.shape))
assert output.shape == (2, 4, 8)
assert torch.count_nonzero(weights.masked_select(~causal_mask[None, :, :])) == 0

## À retenir

- Q et K donnent les scores ; les poids d’attention combinent les valeurs V.
- Le masque causal bloque les tokens futurs avant le softmax.
- La dimension batch permet de traiter plusieurs séquences ensemble.
- Cette classe n’est pas encore multi-têtes et ne contient pas le MLP ni LayerNorm.
- Les données et poids restent synthétiques ; aucun entraînement sur la darija n’a eu lieu.

La prochaine brique sera l’attention multi-têtes, puis l’assemblage du bloc Transformer.